# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Fernando Arreola Treviño
#### **Matrícula:** A01840331

## Objetivo
Notebook de **entrenamiento de modelo** del proyecto MiBici.

La pregunta que me corresponde es la **P2 (clasificación)**: ¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

La lógica que se reutilice se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici**, el sistema de bicicleta pública del Área Metropolitana de Guadalajara. El portal publica:
- **Viajes por mes** (`datos_abiertos_AAAA_MM.csv`) desde diciembre de 2014. Cada registro es un viaje con identificador, datos del usuario, fecha y hora de inicio y fin, y estaciones de origen y destino.
- **Nomenclatura de estaciones** (`nomenclatura_AAAA_MM.csv`) con identificador, nombre, latitud, longitud y capacidad de cada estación.

- Fuente: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. Puede recortarse después con base en los hallazgos.

Se cuenta con lógica implementada para descargar los archivos en un rango especificado y en formato parquet, homologando la codificación y estandarización de fechas (`dataset.py`). Se usó este código para descargar los datos del 2020 al 2026 en la carpeta data/external.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import glob
import pyarrow
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, log_loss,
                             precision_recall_curve, precision_recall_fscore_support)
from sklearn.utils.validation import check_is_fitted


print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("matplotlib  :", matplotlib.__version__)

Python      : 3.11.5
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1
matplotlib  : 3.11.2


## Configuración global


In [2]:
# Rutas del proyecto (el notebook se ejecuta desde notebooks/)
RUTA_PROYECTO = Path.cwd().parent
DATA_DIR = RUTA_PROYECTO / "data" / "processed"
ESTACIONES_DIR = RUTA_PROYECTO / "data" / "raw" / "estaciones"

# Horas sin servicio de MiBici
SIN_SERVICIO = [1, 2, 3, 4]

SEMILLA = 42

HORIZONTE_MIN =120
BIN_MIN = 30
H_BINS = HORIZONTE_MIN // BIN_MIN
ALFA = 0.5

FEATURES = ([f"neto_{k}_relativo" for k in (1, 2, 4, 8)] + [f"{c}_{k}" for c in ("salidas", "entradas") for k in (2, 4)] +
            ["cumsum_dia_relativo", "dpcapacity", "latitude", "longitude", "hora_sin", "hora_cos", "es_fin_semana"])
FEATURES_CON_HISTORICO = FEATURES + ["hist_sin_bicis", "hist_sin_anclajes"]

# Configuracion del modelo
DIAS_TEST = 28          # últimos 28 días de servicio: test (no se usa para decidir nada)
DIAS_VAL = 14           # los 14 días anteriores: solo para elegir los umbrales de decisión
PESO_HISTORICO = 20        # peso (en días) de la frecuencia global en la historia "hasta ayer"
CLASES = ["sin_riesgo", "sin_bicis", "sin_anclajes"]

PARAMS_MODELO = dict(max_iter=150, learning_rate=0.05, min_samples_leaf=500, early_stopping=False, random_state=SEMILLA)



## Carga de datos

Se retoman los datos guardados de la libreta anterior.

In [3]:
def leer_datos():
    parquet_file = os.path.join(DATA_DIR, "p2_datos_cambios_de_capacidad.parquet")
    return pd.read_parquet(parquet_file)

In [4]:
cambios = leer_datos()

## Separación en entrenamiento, validación y test

Se toman los últimos 28 días del dataset como conjunto de prueba. Los 14 días anteriores, son de validación.

In [5]:
def obtener_cortes_temporales(cambios, dias_test, dias_val):
    corte_test = cambios.dia.max() - pd.Timedelta(days=dias_test - 1)
    corte_val = corte_test - pd.Timedelta(days=dias_val)
    return corte_val, corte_test


corte_val, corte_test = obtener_cortes_temporales(cambios, DIAS_TEST, DIAS_VAL)
print("fit: antes de", corte_val.date(), "| val:", corte_val.date(), "a", (corte_test - pd.Timedelta(days=1)).date(),
      "| test: desde", corte_test.date())

fit: antes de 2026-08-20 | val: 2026-08-20 a 2026-09-02 | test: desde 2026-09-03


Se agrega una medición del riesgo histórico de que una estación se quede sin bicicletas o sin anclajes. La idea es: para cada fila (estación, día, hora), ¿qué tan seguido esa estación tuvo alguno de estos riesgos en esa misma hora y tipo de día (entre semana o fin de semana) en los días anteriores? Se calcula como un porcentaje respecto al número de registros anteriores. Esto se usará como feature adicional en el modelo real. Además, el baseline va a predecir este valor para cada riesgo.

A esta medición, se le agrega un parámetro PESO_HISTORICO que se usa para asignarle un peso a esta medición histórica.

In [6]:
class HistoriaHastaAyer(BaseEstimator):
    """Frecuencia de cada clase en los días previos de la misma estación x tipo de día x hora, suavizada hacia la
    frecuencia global del periodo de entrenamiento. No usa el día actual ni el futuro.
    fit(): aprende SOLO la frecuencia global.
    transform(): calcula la historia sobre toda la tabla que reciba"""
 
    def __init__(self, peso_historico=20):
        self.peso_historico = peso_historico
 
    def fit(self, X, y=None):
        self.frec_global_ = {"sin_bicis": float((X["clase"] == 1).mean()),
                             "sin_anclajes": float((X["clase"] == 2).mean())}
        return self
 
    def transform(self, X):
        check_is_fitted(self, "frec_global_")
        
        X = X.sort_values(["estacion", "es_fin_semana", "hora_idx", "dia"]) 
        X["es_sin_bicis"] = (X["clase"] == 1).astype("int8")
        X["es_sin_anclajes"] = (X["clase"] == 2).astype("int8")
        
        # Se agrupa por estación, tipo de día, y hora, y se obtiene un conteo histórico de cada agrupación
        g = X.groupby(["estacion", "es_fin_semana", "hora_idx"], sort=False)
        dias_previos = g.cumcount()
        
        # Para cada clase se riesgo: se calcula la suma de incidencias restando el día actual
        # Esta suma se divide entre el número de días previos, y se obtiene un porcentaje histórico
        # de cuantos días hacia atrás han presentado el riesgo
        # Esto es exactamente lo que va a predecir el modelo baseline, y también serán features del modelo
        # Se agrega una variable M que se usa para asignar peso a este histórico
        f = self.frec_global_
        X["hist_sin_bicis"] = (g["es_sin_bicis"].cumsum() - X["es_sin_bicis"] + self.peso_historico * f["sin_bicis"]) / (dias_previos + self.peso_historico)
        X["hist_sin_anclajes"] = (g["es_sin_anclajes"].cumsum() - X["es_sin_anclajes"] + self.peso_historico * f["sin_anclajes"]) / (dias_previos + self.peso_historico)
        return X
 

historia = HistoriaHastaAyer(peso_historico=PESO_HISTORICO).fit(cambios[cambios.dia < corte_val])    # la frecuencia global sale solo del periodo de entrenamiento
cambios = historia.transform(cambios)
frec_global = historia.frec_global_

Contando con esto, se procede a separar los datos en train, test y val.

In [7]:
# Separar conjuntos de entrenamiento, validación, prueba
def separar_fit_val_test(cambios, corte_val, corte_test):
    fit = cambios[cambios.dia < corte_val]
    val = cambios[(cambios.dia >= corte_val) & (cambios.dia < corte_test)]
    test = cambios[cambios.dia >= corte_test]
    return fit, val, test
 
 
fit, val, test = separar_fit_val_test(cambios, corte_val, corte_test)
y_val, y_test = val.clase.to_numpy(), test.clase.to_numpy()
print("filas fit / val / test:", len(fit), len(val), len(test))
print("prevalencia test:", {CLASES[k]: round(float((y_test == k).mean()), 4) for k in (1, 2)})

filas fit / val / test: 28134216 234504 469694
prevalencia test: {'sin_bicis': 0.008, 'sin_anclajes': 0.0101}


## Entrenamiento: Modelo Baseline

Como se comentó anteriormente, el baseline es la probabilidad de que para una estación, tipo de día (fin de semana o no) y hora, ocurra una clasificación de riesgo a partir del histórico.

In [8]:
# Baseline
class BaselineHistorico(BaseEstimator, ClassifierMixin):
    """Probabilidad de cada clase = su frecuencia histórica 'hasta ayer'. sin_riesgo es lo que sobra."""
 
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1, 2])
        return self
 
    def predict_proba(self, X):
        p_bicis, p_anclajes = X["hist_sin_bicis"].to_numpy(), X["hist_sin_anclajes"].to_numpy()
        return np.column_stack([1 - p_bicis - p_anclajes, p_bicis, p_anclajes])
 
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]
 
 
baseline = BaselineHistorico().fit(fit[FEATURES_CON_HISTORICO], fit.clase)

## Entrenamiento: HistGradientBoostingClassifier

Se escogió un HistGradientBoostingClassifier por su capacidad para entrenar con cantidades masivas de datos. Se entrena a continuación.

In [ ]:
modelo = HistGradientBoostingClassifier(**PARAMS_MODELO).fit(fit[FEATURES_CON_HISTORICO], fit.clase)
assert list(modelo.classes_) == [0, 1, 2]
print("iteraciones del modelo:", modelo.n_iter_)

## Evaluación

In [ ]:
def evaluar_modelo(modelo, val, test, features):
    """Evalúa un modelo ya entrenado. Los umbrales de decisión salen de validación; las métricas, de test.
    Devuelve las probabilidades de test, la clase predicha, los umbrales y las métricas de ese modelo."""
    y_val, y_test = val.clase.to_numpy(), test.clase.to_numpy()
    p_val, p_test = modelo.predict_proba(val[features]), modelo.predict_proba(test[features])
    umbrales = {k: elegir_umbral(y_val == k, p_val[:, k]) for k in (1, 2)}
    pred = decidir_clase(p_test, umbrales)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, labels=[1, 2], zero_division=0)
    metricas = {**calcular_metricas(y_test, p_test),
                "precision_sin_bicis": precision[0], "recall_sin_bicis": recall[0], "f1_sin_bicis": f1[0],
                "precision_sin_anclajes": precision[1], "recall_sin_anclajes": recall[1], "f1_sin_anclajes": f1[1]}
    return {"proba_test": p_test, "pred": pred, "umbrales": umbrales, "metricas": metricas}


def mostrar_evaluacion(nombre, resultado, y_test):
    """Imprime la evaluación de UN modelo: métricas, reporte por clase y matriz de confusión."""
    print(f"\n=== {nombre} ===")
    print("MÉTRICAS EN TEST (PR-AUC y log-loss no dependen del umbral; precisión/recall/F1 sí)")
    print(pd.Series(resultado["metricas"]).round(4).to_string())
    print("\numbrales (elegidos en validación):", {CLASES[k]: round(float(v), 3) for k, v in resultado["umbrales"].items()})
    print(classification_report(y_test, resultado["pred"], target_names=CLASES, digits=3, zero_division=0))
    print("matriz de confusión (filas = real, columnas = predicho):\n",
          confusion_matrix(y_test, resultado["pred"], labels=[0, 1, 2]))

In [ ]:
 # Evaluación en test 
def calcular_metricas(y, p):
    return {"ap_sin_bicis": average_precision_score(y == 1, p[:, 1]),
            "ap_sin_anclajes": average_precision_score(y == 2, p[:, 2]),
            "ap_riesgo": average_precision_score(y > 0, 1 - p[:, 0]),
            "log_loss": log_loss(y, p, labels=[0, 1, 2])}
 

# Se escogen los umbrales de corte con el conjunto de validacion
def elegir_umbral(y_binaria, p):
    precision, recall, umbrales = precision_recall_curve(y_binaria, p)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return umbrales[np.argmax(f1[:-1])]            # el último punto de la curva no tiene umbral asociado
 
 
def decidir_clase(p, umbrales):
    score_bicis, score_anclajes = p[:, 1] / umbrales[1], p[:, 2] / umbrales[2]      # score relativo al umbral de cada clase
    pred = np.zeros(len(p), dtype=int)
    pred[(score_bicis >= 1) & (score_bicis >= score_anclajes)] = 1
    pred[(score_anclajes >= 1) & (score_anclajes > score_bicis)] = 2
    return pred
 
 
def evaluar_modelo(modelo, val, test, features):
    """Evalúa un modelo ya entrenado. Los umbrales de decisión salen de validación; las métricas, de test.
    Devuelve las probabilidades de test, la clase predicha, los umbrales y las métricas de ese modelo."""
    y_val, y_test = val.clase.to_numpy(), test.clase.to_numpy()
    p_val, p_test = modelo.predict_proba(val[features]), modelo.predict_proba(test[features])
    umbrales = {k: elegir_umbral(y_val == k, p_val[:, k]) for k in (1, 2)}
    pred = decidir_clase(p_test, umbrales)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, labels=[1, 2], zero_division=0)
    metricas = {**calcular_metricas(y_test, p_test),
                "precision_sin_bicis": precision[0], "recall_sin_bicis": recall[0], "f1_sin_bicis": f1[0],
                "precision_sin_anclajes": precision[1], "recall_sin_anclajes": recall[1], "f1_sin_anclajes": f1[1]}
    return {"proba_test": p_test, "pred": pred, "umbrales": umbrales, "metricas": metricas}


def mostrar_evaluacion(nombre, resultado, y_test):
    """Imprime la evaluación de un modelo: métricas, reporte por clase y matriz de confusión."""
    print(f"\n=== {nombre} ===")
    print("MÉTRICAS EN TEST (PR-AUC y log-loss no dependen del umbral; precisión/recall/F1 sí)")
    print(pd.Series(resultado["metricas"]).round(4).to_string())
    print("\numbrales (elegidos en validación):", {CLASES[k]: round(float(v), 3) for k, v in resultado["umbrales"].items()})
    print(classification_report(y_test, resultado["pred"], target_names=CLASES, digits=3, zero_division=0))
    print("matriz de confusión (filas = real, columnas = predicho):\n",
          confusion_matrix(y_test, resultado["pred"], labels=[0, 1, 2]))
 
 
resultado_baseline = evaluar_modelo(baseline, val, test, FEATURES_CON_HISTORICO)
mostrar_evaluacion("baseline", resultado_baseline, y_test)

resultado_modelo = evaluar_modelo(modelo, val, test, FEATURES_CON_HISTORICO)
mostrar_evaluacion("modelo", resultado_modelo, y_test)
 


=== baseline ===
MÉTRICAS EN TEST (PR-AUC y log-loss no dependen del umbral; precisión/recall/F1 sí)
ap_sin_bicis              0.2817
ap_sin_anclajes           0.3840
ap_riesgo                 0.3419
log_loss                  0.0615
precision_sin_bicis       0.3153
recall_sin_bicis          0.2841
f1_sin_bicis              0.2989
precision_sin_anclajes    0.4576
recall_sin_anclajes       0.3667
f1_sin_anclajes           0.4071

umbrales (elegidos en validación): {'sin_bicis': 0.218, 'sin_anclajes': 0.199}
              precision    recall  f1-score   support

  sin_riesgo      0.988     0.990     0.989    461168
   sin_bicis      0.315     0.284     0.299      3759
sin_anclajes      0.458     0.367     0.407      4767

    accuracy                          0.979    469694
   macro avg      0.587     0.547     0.565    469694
weighted avg      0.977     0.979     0.978    469694

matriz de confusión (filas = real, columnas = predicho):
 [[456782   2318   2068]
 [  2687   1068      4]
 

El modelo supera al baseline histórico en las tres medidas de ranking: PR-AUC de sin_bicis de 0.282 a 0.328, de sin_anclajes de 0.384 de 0.414 y de riesgo en general de 0.342 a 0.384. Las clases de riesgo son raras (~0.8% y ~1.0% de las ventanas en test), así que ambos modelos están unas 40 veces por encima del azar. El log_loss también mejora (0.0615 → 0.0564, ~8%), por lo que las probabilidades del modelo están al menos tan bien calibradas como las del baseline.

Con los umbrales elegidos en validación, el F1 sube en las dos clases: sin_bicis de 0.299 a 0.348 y sin_anclajes de 0.407 a 0.423. La ganancia en sin_bicis es la más clara. En sin_anclajes el modelo gana sobre todo en recall (0.407 contra 0.367), con una precisión ligeramente menor (0.439 contra 0.458). La matriz de confusión muestra que casi nunca confunde una clase de riesgo con la otra (13 y 14 casos); sus errores son entre riesgo y no riesgo. Detecta cerca de 36% de los casos de sin_bicis y 41% de los de sin_anclajes, con una precisión de 34% y 44%.

In [ ]:
def p2_grafica_matriz_confusion(matriz: np.ndarray, clases: list, titulo: str):
    """Matriz de confusión: color y % por fila (recall de cada clase real), con el conteo debajo.
    Normalizar por fila evita que sin_riesgo, que es ~98% de los casos, aplaste a las clases raras."""
    matriz = np.asarray(matriz)
    por_fila = matriz / matriz.sum(axis=1, keepdims=True)
    figura, ax = plt.subplots(figsize=(6.5, 5.5))
    imagen = ax.imshow(por_fila, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(clases)):
        for j in range(len(clases)):
            color = "white" if por_fila[i, j] > 0.5 else "black"
            ax.text(j, i, f"{por_fila[i, j]:.1%}\n({matriz[i, j]:,})", ha="center", va="center", color=color)
    ax.set_xticks(range(len(clases)), labels=clases)
    ax.set_yticks(range(len(clases)), labels=clases)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.set_title(titulo)
    figura.colorbar(imagen, ax=ax, label="% de la clase real")
    figura.tight_layout()
    return figura

## Registro en MLflow

In [ ]:
import subprocess

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from tc5061_proyecto_equipo07 import config as cfg
from tc5061_proyecto_equipo07 import dataset

MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENTO = "p2_riesgo_estaciones"

# Parámetros del experimento que comparten todos los modelos
PARAMS_COMUNES = {"rango_datos": f"{cfg.MES_INICIO} a {cfg.MES_FIN}", "data_version": dataset.version_datos(),
                  "semilla": SEMILLA, "bin_min": BIN_MIN, "h_bins": H_BINS, "alfa": ALFA,
                  "dias_test": DIAS_TEST, "dias_val": DIAS_VAL, "peso_historico": PESO_HISTORICO,
                  "n_variables": len(FEATURES_CON_HISTORICO), "variables": ",".join(FEATURES_CON_HISTORICO),
                  "datos_desde": str(cambios.dia.min().date()), "datos_hasta": str(cambios.dia.max().date()),
                  "corte_val": str(corte_val.date()), "corte_test": str(corte_test.date()),
                  "filas_fit": len(fit), "filas_val": len(val), "filas_test": len(test)}

# Etiquetas que comparten los runs de P2 (stage, model_family y descripción van por modelo)
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "desconocido"
ETIQUETAS_COMUNES = {"equipo": "07", "pregunta": "P2", "tarea": "clasificacion", "commit": COMMIT,
                     "autor": "Fernando Arreola Treviño", "origen": "notebook"}


def registrar_modelo_en_mlflow(nombre, modelo, resultado, params_modelo, etiquetas_modelo, X_ejemplo, y_test,
                               params_comunes=PARAMS_COMUNES):
    """Un run de MLflow para un modelo: parámetros, métricas de test, matriz de confusión (imagen y conteos) y el modelo.
    nombre: nombre del run ('p2-baseline-historico' o 'p2-hgb', los mismos que usa train.py).
    resultado: lo que devuelve evaluar_modelo (trae predicciones, métricas y umbrales).
    params_modelo: dict con los parámetros propios de ese modelo.
    etiquetas_modelo: stage, model_family y descripción del run.
    X_ejemplo: unas filas de FEATURES_CON_HISTORICO, para guardar la firma (columnas y tipos) del modelo.
    y_test: clases reales de test, para construir la matriz de confusión."""
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENTO)
    umbrales = {f"umbral_{CLASES[k]}": float(v) for k, v in resultado["umbrales"].items()}     # salen de validación
    matriz = confusion_matrix(y_test, resultado["pred"], labels=[0, 1, 2])
    figura = p2_grafica_matriz_confusion(matriz, CLASES, f"Matriz de confusión: {nombre} (test)")
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.set_tags({**ETIQUETAS_COMUNES, **etiquetas_modelo})
        mlflow.log_params({**params_comunes, **params_modelo, **umbrales})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in resultado["metricas"].items()})
        mlflow.log_figure(figura, "p2/matriz_confusion.png")
        mlflow.log_dict({"clases": CLASES, "filas": "clase real", "columnas": "clase predicha", "matriz": matriz.tolist()},
                        "p2/matriz_confusion.json")
        mlflow.sklearn.log_model(modelo, name="modelo",
                                 signature=infer_signature(X_ejemplo, modelo.predict_proba(X_ejemplo)),
                                 input_example=X_ejemplo, pyfunc_predict_fn="predict_proba",
                                 serialization_format="cloudpickle")
    plt.close(figura)
    print(f"{nombre}: run {run.info.run_id} registrado en {MLFLOW_TRACKING_URI}")
    return run.info.run_id


# Se llama por separado, un modelo a la vez:
X_ejemplo = fit[FEATURES_CON_HISTORICO].head(5)
registrar_modelo_en_mlflow("p2-baseline-historico", baseline, resultado_baseline, {"modelo": "baseline"},
                           {"stage": "baseline", "model_family": "frecuencia_historica",
                            "mlflow.note.content": "Frecuencia histórica 'hasta ayer' de cada riesgo (referencia)."},
                           X_ejemplo, y_test)
registrar_modelo_en_mlflow("p2-hgb", modelo, resultado_modelo,
                           {"modelo": "hgb", **{f"modelo_{k}": v for k, v in PARAMS_MODELO.items()},
                            "modelo_n_iter": modelo.n_iter_},
                           {"stage": "candidato", "model_family": "hist_gradient_boosting",
                            "mlflow.note.content": "HistGradientBoostingClassifier con las features y la historia 'hasta ayer'."},
                           X_ejemplo, y_test)

----------------------------------